In [221]:
v_input_sample = """029A
980A
179A
456A
379A"""

v_input = """879A
508A
463A
593A
189A"""

door_keypad = {
    '7': (0, 0),
    '8': (0, 1),
    '9': (0, 2),
    '4': (1, 0),
    '5': (1, 1),
    '6': (1, 2),
    '1': (2, 0),
    '2': (2, 1),
    '3': (2, 2),
    'x': (3, 0),
    '0': (3, 1),
    'A': (3, 2)
}

robot_keypad = {
    'x': (0, 0),
    '^': (0, 1),
    'A': (0, 2),
    '<': (1, 0),
    'v': (1, 1),
    '>': (1, 2)
}

# door keypad <- robot1 ----- robot1_directional_keypad <- robot2 ----- robot2_directional_keypad <- robot2 ----- robot3_directional_keypad <- You

In [170]:
def gen_paths(x_movement, y_movement):
    '''
    For x and y distances from the begining to the end - generate all the variants of DPS (directional pad sequences)
    '''
    
    v_seq = []

    if abs(x_movement) == 0 and abs(y_movement) == 0:
        v_seq = ['A']  
        
    if x_movement >= 1:          
        temp_seq = gen_paths(x_movement - 1, y_movement)
        temp_seq = ['v' + p for p in temp_seq] 
        v_seq.extend(temp_seq)
        
    if x_movement <= -1:
        temp_seq = gen_paths(x_movement + 1, y_movement)
        temp_seq = ['^' + p for p in temp_seq] 
        v_seq.extend(temp_seq)
    
    if y_movement >= 1:
        temp_seq = gen_paths(x_movement, y_movement - 1)
        temp_seq = ['>' + p for p in temp_seq] 
        v_seq.extend(temp_seq)

    elif y_movement <= -1:
        temp_seq = gen_paths(x_movement, y_movement + 1)
        temp_seq = ['<' + p for p in temp_seq] 
        v_seq.extend(temp_seq)

    return v_seq

# # Tests
# for pt in ( (0, 0), (1, 0), (0, 1), (1, 1), (-1, 0), (0, -1), (-1, -1), (-2, -1), (2, 2), (-2, -2) ):
#     print(f'gen_paths{pt} = {gen_paths(pt[0], pt[1])}')


In [173]:
def pass_trough_empty(first_cdn, path, x_coordinate):
    '''
    Checks if the path will pass trough x_coordinates when starts from first_cdn and executes a specific DPS (directional pad sequences).
    If yes returns True, otherwise - False.
    '''
    
    current_cdn = first_cdn
    directions = {'v': (1, 0), '^': (-1, 0), '<': (0, -1), '>': (0, 1), 'A': (0, 0)}
    for d in path:
        current_cdn = (current_cdn[0] + directions[d][0], current_cdn[1] + directions[d][1])
        if current_cdn == x_coordinate:
            return True
    return False        

# # Test
# x_coordinate = (3, 0)

# first_cdn = (3, 1)
# path = '<^A'
# print(f'pass_trough_empty({first_cdn}, \'{path}\', {x_coordinate}) = {pass_trough_empty(first_cdn, path, x_coordinate)}')
# first_cdn = (3, 1)
# path = '^<A'
# print(f'pass_trough_empty({first_cdn}, \'{path}\', {x_coordinate}) = {pass_trough_empty(first_cdn, path, x_coordinate)}')
# first_cdn = (3, 1)
# path = '<'
# print(f'pass_trough_empty({first_cdn}, \'{path}\', {x_coordinate}) = {pass_trough_empty(first_cdn, path, x_coordinate)}')
# first_cdn = (3, 0)
# path = ''
# print(f'pass_trough_empty({first_cdn}, \'{path}\', {x_coordinate}) = {pass_trough_empty(first_cdn, path, x_coordinate)}')

In [174]:
def remove_imposible(first_cdn, v_pair_movements, x_coordinate):
    '''
    For a list of DPS in v_paid_movements removes those which go via x_coordinate
    '''
    
    result = []
    for path in v_pair_movements:
            if not pass_trough_empty(first_cdn, path, x_coordinate):
                result.append(path)
    return result

# # Test
# first_cdn = (3, 2)
# v_pair_movements = ['^^<<A', '^<^<A', '^<<^A', '<^^<A', '<^<^A', '<<^^A']
# print (f"remove_door_imposible({first_cdn}, {v_pair_movements}) =  {remove_imposible(first_cdn, v_pair_movements, door_keypad['x'])}")

In [231]:
def calc_door_code_dps(p_code, debug=False):
    '''
    Returns a list of lists with possible DPS for each symbol of a door keypad code
    '''
    
    v_seq = ''
    
    if len(p_code) < 2:
        return ''
    elif len(p_code) == 2:
        first_cdn = door_keypad[p_code[0]]
        second_cdn = door_keypad[p_code[1]]
        x_movement = second_cdn[0] - first_cdn[0]
        y_movement = second_cdn[1] - first_cdn[1]
        
        if debug: print(f'p_code = {p_code}, first_cdn = {first_cdn}, second_cdn = {second_cdn}, x_movement = {x_movement}, y_movement = {y_movement}')
        
        v_pair_movements = gen_paths(x_movement, y_movement)
        v_pair_movements = remove_imposible(first_cdn, v_pair_movements, door_keypad['x'])
        v_seq = [v_pair_movements]
    elif len(p_code) > 2:
        # The best sequence of the code without the last symbol and the best sequence between the last two symbols
        v_seq = calc_door_code_dps(p_code[:-1], debug) + calc_door_code_dps(p_code[-2:], debug)

    return v_seq

# Tests
# +---+---+---+<>
# | 7 | 8 | 9 |
# +---+---+---+
# | 4 | 5 | 6 |
# +---+---+---+
# | 1 | 2 | 3 |
# +---+---+---+
#     | 0 | A |
#     +---+---+

print (calc_door_code_dps('A029A', True))

p_code = A0, first_cdn = (3, 2), second_cdn = (3, 1), x_movement = 0, y_movement = -1
p_code = 02, first_cdn = (3, 1), second_cdn = (2, 1), x_movement = -1, y_movement = 0
p_code = 29, first_cdn = (2, 1), second_cdn = (0, 2), x_movement = -2, y_movement = 1
p_code = 9A, first_cdn = (0, 2), second_cdn = (3, 2), x_movement = 3, y_movement = 0
[['<A'], ['^A'], ['^^>A', '^>^A', '>^^A'], ['vvvA']]


In [153]:
def get_code_no(p_code):
    '''
    Calculate the numeric value of a door keypad code
    '''
    
    v_number = p_code.replace('A','')
    if v_number:
        return int(v_number)
    else: return 0
    
# # Tests
# print (get_code_no('029A'))
# print (get_code_no('A029A'))
# print (get_code_no('A0A2A9A'))


In [177]:
def calc_directional_pad_dps(dps_code, debug=False):
    '''
    Returns a list of lists with all possible DPS-es for each DPS symbol in the input parameter dps_code
    '''
    v_seq = ''
    
    if len(dps_code) < 2:
        return ''
    elif len(dps_code) == 2:
        first_cdn = robot_keypad[dps_code[0]]
        second_cdn = robot_keypad[dps_code[1]]
        x_movement = second_cdn[0] - first_cdn[0]
        y_movement = second_cdn[1] - first_cdn[1]
        if debug:
            print(f'dps_code = {dps_code}, first_cdn = {first_cdn}, second_cdn = {second_cdn}, x_movement = {x_movement}, y_movement = {y_movement}')
        v_pair_movements = gen_paths(x_movement, y_movement)
        v_pair_movements = remove_imposible(first_cdn, v_pair_movements, robot_keypad['x'])
        v_seq = [v_pair_movements]
    elif len(dps_code) > 2:
        # The best sequence of the code without the last symbol and the best sequence between the last two symbols
        v_seq = calc_directional_pad_dps(dps_code[:-1], debug) + calc_directional_pad_dps(dps_code[-2:], debug)

    return v_seq

# # Test
# #     +---+---+
# #     | ^ | A |
# # +---+---+---+
# # | < | v | > |
# # +---+---+---+
# dps_code = 'A<A^A>^^AvvvA'
# print(f'dps_code={dps_code}: {calc_directional_pad_dps(dps_code, debug=False)}')
# # expected result: v<<A >>^A <A >A vA <^A A >A <vA A A >^A

# for dps_code in ('v<<A', '<v<A'):
#     print(f"\n\ndps_code={'A' + dps_code}: [ {calc_directional_pad_dps('A' + dps_code, debug=False)} ]")



In [148]:
def sequences_list_expansion(v_sequences, debug=False):
    '''
    Takes a list of lists of DPS-es and combines each DPS in the inner lists with each other inner DPS in the other lists
    '''
    
    seqs = v_sequences[0]

    if debug: print(f'initial seqs = {seqs}')
    
    for position_seqs in v_sequences[1:]:
        temp_seq = []
        for path in position_seqs:
            ext = [s + path for s in seqs]  
            if debug: print(f'    inner loop ext = {ext}')
            temp_seq.extend(ext)
        if debug: print(f'assigning temp_seqs = {temp_seq}\n')
        seqs = temp_seq
        
    return seqs

# # Test
# p = [['v<<A', '<v<A'], ['>^>A', '>>^A'], ['<A'], ['>A'], ['vA'], ['^<A', '<^A'], ['A'], ['>A'], ['v<A', '<vA'], ['A'], ['A'], ['^>A', '>^A']]
# print(f'sequences_list_expansion(p) = {sequences_list_expansion(p)}')

In [205]:
def reduce_k3_dps(k3_dict):
    k2_dps = list(k3_dict.keys())
    temp_k2_dps = k2_dps[0]
    temp_k3_dps = k3_dict[temp_k2_dps][0]
    min_len = len(temp_k3_dps)   
    
    for k in k3_dict:
        for s in k3_dict[k]:
            if len(s) < min_len:
                min_len = len(s)
                temp_k2_dps = k
                temp_k3_dps = s
                
    return (k, s, min_len)

In [212]:
# door keypad <- r1 ----- robot1_directional_keypad [K1] <- r2 ----- r2_directional_keypad [K2] <- r3 ----- r3_directional_keypad [K3] <- You

def calc_complexity_one_code(code):
    '''
    calculates the complexity of a single code
    '''
    
    initial_pos = 'A'
    k1_ll_dps = calc_door_code_dps(initial_pos + code) # list of lists of possible dps for a single symbol in the code
    k1_dps = sequences_list_expansion(k1_ll_dps)

    k2_dict = dict()
    
    for s in k1_dps:
        k2_ll_dps = calc_directional_pad_dps(initial_pos + s)
        k2_dps = sequences_list_expansion(k2_ll_dps)
        k2_dict[s] = k2_dps

    k3_dict = dict()

    for k1_seq in k2_dict:
        for k2_sec in k2_dict[k1_seq]:
            k3_ll_dps = calc_directional_pad_dps(initial_pos + k2_sec)
            k3_dps = sequences_list_expansion(k3_ll_dps)
            k3_dict[k2_sec] = k3_dps

    min_k2_dps, min_len_k3_dps, min_len = reduce_k3_dps(k3_dict)
    
    complexity = min_len * get_code_no(code)
    return complexity

# # Test
# for code in ['029A','980A', '179A', '456A', '379A']:
#     print(f'code = {code} has complexity = {calc_complexity_one_code(code)}')

# # code = 029A has complexity = 1972
# # code = 980A has complexity = 58800
# # code = 179A has complexity = 12172
# # code = 456A has complexity = 29184
# # code = 379A has complexity = 24256
# # 1972 + 58800 + 12172 + 29184 + 24256


In [214]:
#Step 2 - based on a list of codes, calculates the total complexity ot all of them

def calc_complexities_sum(l_input):
    s = 0 # Calculator
    for code in l_input:
        s += calc_complexity_one_code(code)  # Step 2.1
    return s

# test
print(calc_complexities_sum(l_input))

126384


In [222]:
# Step 1 - simply get the task definition and make a list from it

def get_input(p_input_str):

    l_input = p_input_str.split()
    return l_input

In [223]:
# Get the input, calculate its complexity and print the result

#l_input = get_input(v_input_sample) # Step 1
l_input = get_input(v_input) # Step 1

cpxt_sum = calc_complexities_sum(l_input) # Step 2
print (cpxt_sum)


188384


# Part 2

In [611]:
# # Tests
# # +---+---+---+<>
# # | 7 | 8 | 9 |
# # +---+---+---+
# # | 4 | 5 | 6 |
# # +---+---+---+
# # | 1 | 2 | 3 |
# # +---+---+---+
# #     | 0 | A |
# #     +---+---+



v_input_sample_l = ['029A', '980A', '179A', '456A', '379A']

v_input_l = ['879A', '508A', '463A', '593A', '189A']

door_tc = {
    'A0': '<A', # no alt.
    'A1': '^<<A', # original: ^<<A, alsternative <^< IS WORSE
    #'A2': '<^A', # original <^A, alternative ^<A NO DIFFERENCE
    'A3': '^A', # no alt.
    'A4': '^^<<A', # original ^^<<A, alt. ^<<^A is WORSE
    'A5': '<^^A', # original <^^A, alt. ^^<A is WORSE
    #'A6': '^^A', # no alt.
    #'A7': '^^^<<A', # original ^^^<<A, alternatives <^<^^A NO DIFFERENCE
    'A8': '<^^^A', # original <^^^A alternative ^^^<A is worse
    'A9': '^^^A', # no alt.
    '02': '^A', # no alt.
    '08': '^^^A', # no alt.
    '0A': '>A', # no alt.
    '17': '^^A', # no alt.
    '18': '^^>A', # original ^^>A, alternatives >^^A ^>^A are WORSE
    '29': '>^^A', # original >^^A, , alternatives, ^^>A  ^>^ are WORSE
    '37': '<<^^A', # original <<^^A, alternative ^^<<A is worse
    '3A': 'vA', # no alt.
    '45': '>A', # no alt.
    '46': '>>A', # no alt.
    '50': 'vvA', # no alt.
    '56': '>A', # no alt.
    '59': '^>A', # no alt.
    '6A': 'vvA', # no alt.
    '63': 'vA', # no alt.
    '79': '>>A', # no alt.
    '80': 'vvvA', # no alt.
    '87': '<A', # no alt.
    '89': '>A', # no alt.
    '8A': 'vvv>A', # 8A vvv>A  - alternatives >vvvA, vv>vA, v>vvA are worse than vvv>A
    '93': 'vvA', # no alt.
    '98': '<A', # no alt.
    '9A': 'vvvA' # no alt.
}

# #     +---+---+
# #     | ^ | A |
# # +---+---+---+
# # | < | v | > |
# # +---+---+---+

dps_tc = {
    'A^': '<A',
    'A>': 'vA',
    'Av': '<vA', #'v<'
    'A<': 'v<<A', # v<<
    'AA': 'A',
    '^^': 'A',
    '^>': 'v>A',
    '^<': 'v<A',
    '^A': '>A',
    '>>': 'A',
    '>v': '<A',
    '>^': '<^A', # '^<A'
    '>A': '^A',
    'vv': 'A',
    'v<': '<A',
    'v>': '>A',
    'vA': '>^A',
    '<<': 'A',
    '<^': '>^A',
    '<v': '>A',
    '<A': '>>^A'
}

dps_tc_n = {
    'A^': '<A',
    'A>': 'vA',
    'Av': '<vA', #'v<'
    'A<': 'v<<A', # v<<
    'AA': 'A',
    '^^': 'A',
    '^>': 'v>A',
    '^<': 'v<A',
    '^A': '>A',
    '>>': 'A',
    '>v': '<A',
    '>^': '<^A', # '^<A'
    '>A': '^A',
    'vv': 'A',
    'v<': '<A',
    'v>': '>A',
    'vA': '>^A',
    '<<': 'A',
    '<^': '>^A',
    '<v': '>A',
    '<A': '>>^A'
}

dps_next_hop = {
    #'A' : [''],
    'A^': ['A<', '<A'],
    'A>': ['Av', 'vA'],
    'Av': ['A<', '<v', 'vA'],  # ['Av', 'v<', '<A'],
    'A<': ['Av', 'v<', '<<', '<A'], # ['Av', 'v<', '<<', '<A']
    'AA': ['AA'],
    '^^': ['AA'],
    '^>': ['Av', 'v>', '>A'],
    '^<': ['Av', 'v<', '<A'],
    '^A': ['A>', '>A'],
    '>>': ['AA'],
    '>v': ['A<', '<A'],
    '>^': ['A<', '<^', '^A'], # ['A^', '^<', '<A']
    '>A': ['A^', '^A'],
    'vv': ['AA'],
    'v<': ['A<', '<A'],
    'v>': ['A>', '>A'],
    'vA': ['A>', '>^', '^A'],
    '<<': ['AA'],
    '<^': ['A>', '>^', '^A'],
    '<v': ['A>', '>A'],
    '<A': ['A>', '>>', '>^', '^A']
}

dps_next_hop_n = {
    #'A' : [''],
    'A^': ['A<', '<A'], # no alt.
    'A>': ['Av', 'vA'], # no alt.
    'Av': ['A<', '<v', 'vA'],  # alternative ['Av', 'v<', '<A'] is WORSE
    'A<': ['Av', 'v<', '<<', '<A'], # alternative ['A<', '<v', 'v<', '<A'] is WORSE
    'AA': ['AA'], # no alt.
    '^^': ['AA'], # no alt.
    '^>': ['Av', 'v>', '>A'], #alternative ['A>', '>v', '>vA'] is WORSE
    '^<': ['Av', 'v<', '<A'], # no alt.
    '^A': ['A>', '>A'], # no alt.
    '>>': ['AA'], # no alt.
    '>v': ['A<', '<A'], # no alt.
    '>^': ['A<', '<^', '^A'], # alternative ['A^', '^<', '<A'] is WORSE
    '>A': ['A^', '^A'], # no alt.
    'vv': ['AA'], # no alt.
    'v<': ['A<', '<A'], # no alt.
    'v>': ['A>', '>A'], # no alt.
    'vA': ['A^', '^>', '>A'], # alternative ['A>', '>^', '^A'] is WORSE
    '<<': ['AA'], # no alt.
    '<^': ['A>', '>^', '^A'],
    '<v': ['A>', '>A'], # no alt.
    '<A': ['A>', '>>', '>^', '^A']
}

#       +-----+-----+
#       | ^ 4 | A 5 |
# +-----+-----+-----+
# | < 1 | v 2 | > 3 |
# +-----+-----+-----+

# dps_tc_num = {
#     'A^': 54, -> '<A' = 15
#     'A>': 53, -> 'vA' = 25
#     'Av': 52, -> 'v<A' = 215
#     'A<': 51, -> 'v<<A' = 2115, 21 11 15 -> 15 5 3345
#     'AA': 55, -> 'A' = 5
#     '^^': 44, -> 'A' = 5
#     '^>': 43, -> 'v>A' = 235
#     '^<': 41, -> 'v<A' = 215
#     '^A': 45, -> '>A' = 35
#     '>>': 33, -> 'A', = 5
#     '>v': 32, -> '<A', = 15
#     '>^': 34, -> '^<A', = 415
#     '>A': 35, -> '^A', = 45
#     'vv': 22, -> 'A', = 5
#     'v<': 21, -> '<A', = 15
#     'v>': 23, -> '>A', = 35
#     'vA': 25, -> '>^A', = 345
#     '<<': 11, -> 'A', = 5
#     '<^': 14, -> '>^A', = 345
#     '<v': 12, -> '>A', = 35
#     '<A': 15  -> '>>^A' = 3345
# }

# dps_tc_num = {
#     'A^': 15,
#     'A>': 25,
#     'Av': 215,
#     'A<': 2115,
#     'AA': 5,
#     '^^': 5,
#     '^>': 235,
#     '^<': 215,
#     '^A': 35,
#     '>>': 5,
#     '>v': 15,
#     '>^': 415,
#     '>A': 45,
#     'vv': 5,
#     'v<': 15,
#     'v>': 35,
#     'vA': 345,
#     '<<': 5,
#     '<^': 345,
#     '<v': 35,
#     '<A': 3345
# }

In [577]:
def gen_door_seq (p_code):
    acc = ''
    for i in range(len('A' + p_code) - 1):
        acc += door_tc[('A' + p_code)[i: i+2]]
    return (acc)

# Test 
gen_door_seq ('029A')

# 029A -> '<A^A>^^AvvvA'

'<A^A>^^AvvvA'

In [567]:
def gen_dps_seq (p_code, p_dps_tc):
    acc = ''
    code_str = 'A' + p_code
    for i in range(len(code_str) - 1):
        itteration_code = ('A' + p_code)[i: i+2]
        acc += p_dps_tc[itteration_code]
         
        if itteration_code in cases_dict:
            code_cnt = cases_dict[itteration_code]
        else:
            code_cnt = 0

    return (acc)

# Test 
gen_dps_seq ('v<<A>>^A<A>AvA<^AA>A<vAAA>^A', dps_tc_n)
# 'v<<A>>^A<A>A<AAv>A^Av<AAA>^A'

# 029A -> '<A^A>^^AvvvA' ->  v<<A>>^A<A>AvA<^AA>A<vAAA>^A -> <vA<AA>>^AvAA<^A>A<v<A>>^AvA^A<vA>^A<v<A>^A>AAvA^A<v<A>A>^AAAvA<^A>A
#                            v<<A>>^A<A>AvA<^AA>A<vAAA>^A -> <vA<AA>>^AvAA<^A>A<v<A>>^AvA^A<vA>^A<v<A>^A>AAvA^A<v<A>A>^AAAvA<^A>A

'<vA<AA>>^AvAA<^A>A<v<A>>^AvA^A<vA>^A<v<A>^A>AAvA^A<v<A>A>^AAAvA<^A>A'

In [392]:
def trans_dps_seq_to_dict (p_code, debug=False):
    
    d = {}
    v_code = 'A' + p_code
    print(v_code) if debug else None
    for i in range(len(v_code) - 1):
        key = v_code[i:i+2]
        
        if key in d:
            val = d[key] + 1
        else:
            val = 1
    
        d.update({key: val})
        print (f'i = {i},  key = {key}, val = {val}') if debug else None

    return d

# # Test
# trans_dps_seq_to_dict ('<A^A^^>AvvvA')

# # 'A<': 1,
# # '<A': 1,
# # 'A^': 2,
# # '^A': 1,
# # '^^': 1,
# # '^>': 1,
# # '>A': 1,
# # 'Av': 1,
# # 'vv': 2,
# # 'vA': 1

In [393]:
def len_dict(p_dict):
    acc = 0
    for k in p_dict:
        if k != 'A':
            acc += p_dict[k]
    return acc

# Test

len_dict({'A<': 1,
 '<A': 1,
 'A^': 2,
 '^A': 1,
 '^^': 1,
 '^>': 1,
 '>A': 1,
 'Av': 1,
 'vv': 2,
 'vA': 1})

12

In [563]:
def gen_dps_next_dict (cases_dict, p_dps_next_hop,  debug=False):

    new_cases_dict = dict(cases_dict)
    print(f'Initial new_cases_dict = {new_cases_dict}\n') if debug else None
    
    for k in p_dps_next_hop:        
        if k not in new_cases_dict:
            new_cases_dict.update({k: 0})        

    print (new_cases_dict) if debug else None
    
    for i in cases_dict:        

        itteration_code_list = p_dps_next_hop[i]
        
        print(f'Reduction: new_cases_dict[{i}]: new_cases_dict[{i}] = {new_cases_dict[i]} - cases_dict[{i}] = {cases_dict[i]}') if debug else None
        new_cases_dict[i] = new_cases_dict[i] - cases_dict[i]
        

        print(f'i = {i}, itteration_code_list = {itteration_code_list}') if debug else None

        for iteration_code in itteration_code_list:
            code_cnt = new_cases_dict[iteration_code]       

            print(f'     new_cases_dict[{iteration_code}]: code_cnt =  {code_cnt} + cases_dict[{i}] = {cases_dict[i]}') if debug else None
            new_cases_dict.update({iteration_code: code_cnt + cases_dict[i]})
            

    # if first_move == 'A<':
    #     v_down_left = new_cases_dict['v<']
    #     v_left_left = new_cases_dict['<<']
    #     v_left_A = new_cases_dict['<A']
        
    #     new_cases_dict.update({'v<': v_down_left + 1})
    #     new_cases_dict.update({'<<': v_left_left + 1})
    #     new_cases_dict.update({'<A': v_left_A + 1})
        
    # else:
    #     v_down_left = new_cases_dict['v<']
    #     v_left_A = new_cases_dict['<A']
        
    #     new_cases_dict.update({'v<': v_down_left + 1})
    #     new_cases_dict.update({'<A': v_left_A + 1})        
        
    return new_cases_dict

# # Test 
# # 379A

# r1 = gen_door_seq ('379A')

# r1_dict = trans_dps_seq_to_dict(r1)

# r2 = gen_dps_seq(r1)

# # cases_dict = {
# #  'A<': 1,
# #  '<A': 1,
# #  'A^': 2,
# #  '^A': 1,
# #  '^^': 1,
# #  '^>': 1,
# #  '>A': 1,
# #  'Av': 1,
# #  'vv': 2,
# #  'vA': 1}

# # cases_dict = {'A<': 2, '<A': 4, 'A^': 1, '^A': 3, '^^': 0, '^>': 0, '>A': 2, 'Av': 3, 'vv': 0, 'vA': 0, 'A': 3, 'A>': 3, 'AA': 3, '^<': 0, '>>': 1, '>v': 0, '>^': 2, 'v<': 2, 'v>': 1, '<<': 1, '<^': 0, '<v': 0}

# r2_dict = gen_dps_next_dict (r1_dict)

# print(r2)
# print(gen_dps_seq(r2))
# print(len(gen_dps_seq(r2)))

# print(f'\n\nCalculation od gen_dps_next_dict')
# d = gen_dps_next_dict (r2_dict, True)

# print(f'End calculation od gen_dps_next_dict\n\n')
# print(d)
# print(len_dict(d))
# # 'v<<A>>^A<A>A<AAv>A^Av<AAA>^A'

In [572]:
def gen_dps_seq_power (p_dict, power):

    acc = ''
    
    if power == 1:
        return  gen_dps_next_dict(p_dict, dps_next_hop_n)
    else:
        return gen_dps_seq_power(gen_dps_next_dict(p_dict, dps_next_hop_n), power - 1)

# # Test
# r1_dict = trans_dps_seq_to_dict ('<A^A^^>AvvvA')
# rn_dict = gen_dps_seq_power (r1_dict, 2)
# print(rn_dict)
# print(len_dict(rn_dict))

# #len(gen_dps_seq_power ('<A^A^^>AvvvA', 2))

In [613]:
acc = 0
for p_code in v_input_l: #['029A']: # ['029A']: #  ['980A']: 179A  456A  379A  v_input_sample_l
    r1 = gen_door_seq(p_code) 
    r1_dict = trans_dps_seq_to_dict (r1)
    r2_dict = gen_dps_next_dict(r1_dict, dps_next_hop_n)
    rn_dict = gen_dps_seq_power(r2_dict, 1)
    cmx = get_code_no(p_code) * len_dict(rn_dict)
    acc += cmx
    print(f'Calculation of {p_code} is ready. The complexity is {cmx}')
print(f'acc = {acc}')

Calculation of 879A is ready. The complexity is 61530
Calculation of 508A is ready. The complexity is 36576
Calculation of 463A is ready. The complexity is 32410
Calculation of 593A is ready. The complexity is 43882
Calculation of 189A is ready. The complexity is 13986
acc = 188384
